In [ ]:
import sqlite3
class SQLiteResult:
    @staticmethod
    def read_sql_query(sql, connection):
        cursor = connection.execute(sql)
        columns = [column[0] for column in cursor.description]
        return [dict(zip(columns, row)) for row in cursor.fetchall()]

pd = SQLiteResult

con=sqlite3.connect(r"C:\Users\muham\OneDrive\Desktop\SEMSTER 3\project 3.1_sem 3\data\raw\ipl.db")
def q(sql):
    return pd.read_sql_query(sql,con)

ImportError: DLL load failed while importing internals: An Application Control policy has blocked this file.

In [ ]:
def run_sql_file(path):
    with open(path, 'r') as f:
        script = f.read()
        con.executescript(script)
        con.commit()
        print('ran',path)


In [ ]:
q("""
-- what is the strike rate of every batter in the death overs

SELECT batter,
ROUND(
    100.0 * SUM(batter_runs) / SUM(
        CASE
            WHEN is_wide_ball = 0 AND is_no_ball = 0 THEN 1
            ELSE 0
        END
    ),
    2
) AS strike_rate

FROM deliveries
WHERE over_number >= 15 AND is_super_over = 0
GROUP BY batter;""")

In [ ]:
run_sql_file(r'C:\Users\muham\OneDrive\Desktop\SEMSTER 3\project 3.1_sem 3\sql\09_analysis_views.sql')

In [ ]:
q("""
SELECT *
 FROM v_ball;
""")

In [ ]:
q(""" SELECT phase, count(*) FROM v_ball GROUP BY phase;""")

In [ ]:
run_sql_file(r'C:\Users\muham\OneDrive\Desktop\SEMSTER 3\project 3.1_sem 3\sql\09_analysis_views.sql ')

In [ ]:
q("""SELECT * FROM v_innings;""")

In [ ]:
run_sql_file(r'C:\Users\muham\OneDrive\Desktop\SEMSTER 3\project 3.1_sem 3\sql\09_analysis_views.sql ')

In [ ]:
q("""
SELECT * FROM v_match_totals;""")

In [ ]:
q("""
SELECT * FROM v_ball;""")

In [ ]:
q("""SELECT count(*) FROM v_match_totals;""")

In [ ]:
q("""SELECT COUNt(*) FROM v_innings ;""")

In [ ]:
q("""SELECT phase,COUNT(*)FROM v_ball;""")

In [ ]:
q("""
-- Give the list of highest runs scored in IPL history
--Max 5
SELECT batter,
       SUM(batter_runs) AS total_runs
FROM v_ball
GROUP BY batter
ORDER BY total_runs DESC
LIMIT 5;
""")

In [ ]:
q("""
-- Give the list of highest sixes scored in IPL history
--Max 10
SELECT batter,
       COUNT(*) AS total_sixes
FROM v_ball
where batter_runs=6
GROUP BY batter
ORDER BY total_sixes DESC
LIMIT 10;
""")

In [ ]:
q("""
--highest ball faced by a player,total runs,strike rate
--strike rate = (total runs/total balls)*100
select batter,
       sum(is_legal) as balls,
         sum(batter_runs) as runs,
            round(100.0 * sum(batter_runs) / sum(is_legal), 2) as strike_rate
            from v_ball
            group by batter
            order by balls desc
            limit 10;
            """)

In [ ]:
q("""
--batter who scored the highest share of runs in boundary

select batter,
         sum(batter_runs) as runs,
         round(100.0 *sum(case when batter_runs in (4,6) 
         then batter_runs
         else 0
         end)/sum(batter_runs),2)as boundary_perc

from v_ball
group by batter
having runs>2000
order by boundary_perc desc
limit 10; 
""")

In [ ]:
q("""
--find an ideal middle-order batter who can lift the run rate in over 7 to 15.

  select batter,sum(is_legal) as balls , sum (batter_runs) as runs,
            Round(100* sum(batter_runs)/sum(is_legal),2) as run_rate
from v_ball
where phase = 'Middle'
group by batter
HAVING balls > 500
order by run_rate desc
limit 20;
""")

In [ ]:
q("""
SELECT batter,
       SUM(is_legal) AS balls,
       SUM(batter_runs) AS runs,
       ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS strike_rate
FROM v_ball
where phase = 'Death'
GROUP BY batter
HAVING balls >= 50
ORDER BY strike_rate DESC
LIMIT 10;
""")

In [ ]:
q("""
SELECT batter,
       SUM(is_legal) AS balls,
       SUM(batter_runs) AS runs,
       ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS strike_rate
FROM v_ball
where phase = 'Powerplay'
GROUP BY batter
HAVING balls >= 50
ORDER BY strike_rate DESC
LIMIT 10;
""")

In [ ]:
q("""
-- who is the most reliable in a chase - strike  rate batting second ?

SELECT 
    batter,
    SUM(batter_runs) AS runs,
    SUM(is_legal) AS balls,
    ROUND(100.0 * SUM(batter_runs) / SUM(is_legal), 2) AS strike_rate

FROM v_ball
WHERE innings = 2
GROUP BY batter
HAVING balls > 500
ORDER BY strike_rate DESC
LIMIT 10;

""")

In [ ]:
q("""
SELECT
    batter,
    SUM(is_legal) AS balls,
    SUM(batter_runs) AS runs,
    ROUND(100.0 * SUM(batter_runs) / NULLIF(SUM(is_legal), 0), 2) AS strike_rate
FROM v_ball
WHERE phase = 'Death'
GROUP BY batter
HAVING SUM(is_legal) >= 100
ORDER BY strike_rate DESC
LIMIT 10;
""")

In [ ]:
q("""
-- Which batters are hardest to bowl a dot ball to in the death overs?

SELECT
    batter,
    SUM(is_legal) AS balls,
    SUM(CASE
        WHEN batter_runs = 0 AND is_legal = 1 THEN 1
        ELSE 0
    END) AS dot_balls,
    ROUND(100.0 * SUM(CASE
        WHEN batter_runs = 0 AND is_legal = 1 THEN 1
        ELSE 0
    END) / NULLIF(SUM(is_legal), 0), 2) AS dot_pct
FROM v_ball
WHERE phase = 'Death'
GROUP BY batter
HAVING SUM(is_legal) >= 500
ORDER BY dot_pct ASC
LIMIT 5;
""")

In [ ]:
q("""
--how do runs and wickets differ across the three phases of an innings?


select phase,sum(is_legal)as balls,sum(total_runs),
round(6.0*sum(total_runs)/sum(is_legal),2)as runs_per_over,

round(100.0*sum(is_wicket)/sum(is_legal),2)as wicket_perc

from v_ball
group by phase;

""")

In [ ]:
q("""
--which over of the innings is the most expensive?


select over_number+1 as overs,
round (6.0*sum(total_runs)/sum(is_legal),2)as runs_per_over
from v_ball
group by over_number
order by runs_per_over desc;


""")

In [ ]:
q("""
-- What is the average score at the end of the powerplay?

SELECT AVG(pp_runs) AS avg_powerplay_score
FROM (
    SELECT
        match_id,
        innings,
        SUM(total_runs) AS pp_runs
    FROM v_ball
    WHERE phase = 'Powerplay'
      AND innings IN (1, 2)
    GROUP BY match_id, innings
);
""")

In [ ]:
q("""
--which over sees the most wikets fall?

SELECT
    over_number + 1 AS over,
    SUM(is_wicket) AS wickets
FROM v_ball
GROUP BY over_number
ORDER BY over_number DESC;
""")

In [ ]:
q("""
SELECT bowler,SUM(bowler_wicket) AS wicket 
FROM v_ball
GROUP BY bowler
ORDER BY wicket DESC;
""")

In [ ]:
q("""
    SELECT 
        bowler,
        COUNT(*) AS legal_balls,
        SUM(total_runs) AS runs_conceded,
        ROUND(6.0 * SUM(total_runs) / COUNT(*), 2) AS economy
    FROM v_ball
    WHERE is_wide_ball = 0 
    AND is_no_ball = 0
    GROUP BY bowler
    HAVING COUNT(*) >= 500
    ORDER BY economy ASC
    LIMIT 10;
""")

In [ ]:
q("""SELECT bowler, SUM(is_legal) AS balls,
 ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2) AS b_economy
FROM v_ball
GROUP BY bowler
HAVING balls > 500
ORDER BY b_economy
LIMIT 10;
""")

In [ ]:
q(""" 
SELECT bowler,SUM(is_legal) AS total_balls,SUM(bowler_wicket) AS total_wicket,

              ROUND(1.0*SUM(is_legal)/SUM(bowler_wicket),2) AS balls_per_wicket

FROM v_ball
GROUP BY bowler 
having total_wicket>50
ORDER BY balls_per_wicket;


""")

In [ ]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(CASE WHEN batter_runs IN (4, 6) THEN 1 ELSE 0 END) AS boundaries,
    ROUND(
        100.0 * SUM(CASE WHEN batter_runs IN (4, 6) THEN 1 ELSE 0 END)
        / SUM(is_legal),
        2
    ) AS boundary_perc
FROM v_ball
GROUP BY bowler
HAVING SUM(is_legal) >= 1000
ORDER BY boundary_perc ASC
LIMIT 10;
""")

In [ ]:
q("""

SELECT
    bowler,
    SUM(CASE WHEN is_legal = 1 AND batter_runs = 0 THEN 1 ELSE 0 END) AS dot_balls,
    ROUND(
        100.0 * SUM(CASE WHEN is_legal = 1 AND batter_runs = 0 THEN 1 ELSE 0 END)
        / NULLIF(SUM(is_legal), 0), 2
    ) AS dot_ball_percent
FROM v_ball
GROUP BY bowler
ORDER BY dot_balls DESC
LIMIT 10;

""")

In [ ]:
q("""

--how do bolwers actually take their wickets?

select wicket_kind,count(*)as n
from v_ball
where is_wicket=1
group by wicket_kind
order by n desc;


""")

In [ ]:
q("""

SELECT CASE WHEN bowler_type LIKE                       -- spin or pace 
       '%Legbreak%' OR bowler_type LIKE 
       '%Offbreak%' OR bowler_type LIKE 
       '%Orthodox%' OR bowler_type LIKE 
       '%Wrist spin%' OR bowler_type LIKE 
       '%Googly%' THEN 'Spin' ELSE 'Pace' 
       END AS style,                                    -- branches of th 
       SUM(is_legal) AS balls,                          -- sample size 
       ROUND(6.0*SUM(bowler_runs)/SUM(is_legal),2)      -- runs per over 
              AS economy, 
       ROUND(100.0*SUM(bowler_wicket)/SUM(is_legal),2)  -- bowler's wicke 
              AS wicket_pct 
FROM   v_ball                                           -- one row = one 
WHERE  bowler_type IS NOT NULL AND                      -- drop the blanks 
       TRIM(bowler_type) <> '' 
GROUP  BY style;                                        -- one row per gr

""")

In [ ]:
q("""
SELECT
    CASE
        WHEN bowler_type LIKE 'Left%'
        THEN 'Left arm'
        ELSE 'Right arm'
    END AS arm,

    batsman_type,

    ROUND(
        6.0 * SUM(bowler_runs) / SUM(is_legal),
        2
    ) AS economy,

    ROUND(
        100.0 * SUM(bowler_wicket) / SUM(is_legal),
        2
    ) AS wicket_pct

FROM v_ball

WHERE TRIM(COALESCE(bowler_type, '')) <> ''
  AND batsman_type IS NOT NULL

GROUP BY arm, batsman_type;
""")

In [ ]:
q("""
SELECT
    bowler,
    SUM(bowler_wicket) AS wicket,
    ROUND(
        100.0 * SUM(bowler_wicket) / SUM(is_legal),
        2
    ) AS wicket_perc
FROM v_ball
WHERE bowler_type LIKE '%Legbreak%'
   OR bowler_type LIKE '%Offbreak%'
   OR bowler_type LIKE '%Orthodox%'
   OR bowler_type LIKE '%Wrist spin%'
   OR bowler_type LIKE '%Googly%'
GROUP BY bowler
HAVING SUM(is_legal) >= 1000
ORDER BY wicket DESC
LIMIT 10;
""")

In [ ]:
q("""

--we need a death-over bowler.who is cheapest in over 16 to 20?
--bowler,balls,bowler_runs,economy

SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(bowler_runs) AS bowler_runs,
    ROUND(6.0 * SUM(bowler_runs) / SUM(is_legal), 2) AS economy
FROM v_ball
WHERE over_number BETWEEN 15 AND 19
GROUP BY bowler
HAVING SUM(is_legal) >=300
ORDER BY economy ASC
LIMIT 10;

""")

In [ ]:
q("""
SELECT
    bowler,
    SUM(bowler_wicket) AS wickets,
    ROUND(
        100.0 * SUM(bowler_wicket) / SUM(is_legal),
        2
    ) AS wicket_perc
FROM v_ball
WHERE over_number BETWEEN 0 AND 5
GROUP BY bowler
ORDER BY wickets DESC
LIMIT 10;
""")

In [ ]:
q("""
SELECT
    bowler,
    SUM(is_legal) AS balls,
    SUM(is_legal AND total_runs = 0) AS dot_balls,
    ROUND(100.0 * SUM(is_legal AND total_runs = 0) / SUM(is_legal), 2) AS dot_perc
FROM v_ball
WHERE phase = 'Death'
GROUP BY bowler
HAVING balls >= 300
ORDER BY dot_perc DESC
LIMIT 10;
""")

In [ ]:
run_sql_file(r'C:\Users\muham\OneDrive\Desktop\SEMSTER 3\project 3.1_sem 3\sql\08_matches_clean.sql')

In [ ]:
q("""

SELECT
    COUNT(*) AS decisive_matches,
    SUM(
        CASE
            WHEN toss_winner = match_winner THEN 1
            ELSE 0
        END) AS toss_winner_won,
 ROUND( 100.0 * SUM(
            CASE
                WHEN toss_winner = match_winner THEN 1
                ELSE 0
            END
        ) / COUNT(*),2) AS pct

FROM matches_clean
WHERE result = 'win';

""")

In [ ]:
q("""
SELECT
    LOWER(toss_decision) AS decision,
    COUNT(*) AS total_tosses,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM matches_clean), 2) AS pct
FROM matches_clean
GROUP BY LOWER(toss_decision);
""")

In [ ]:
q("""select count(*)from matches_clean;""")

In [ ]:
q("""
SELECT
    toss_winner,
    COUNT(*) AS toss_won
FROM matches_clean
GROUP BY toss_winner
ORDER BY toss_won DESC;
""")

In [ ]:
q("""select*from v_match_totals;""")

In [ ]:
q("""

SELECT ROUND(AVG(first_innings_runs), 2) AS average_first_innings_score
FROM v_match_totals;

""")

In [ ]:
q("""
SELECT
    innings,
    ROUND(6.0 * SUM(total_runs) / SUM(is_legal), 2) AS run_rate
FROM v_ball
WHERE innings IN (1, 2)
GROUP BY innings
ORDER BY innings;
""")

In [ ]:
q("""

select * FROM v_innings;

""")

In [ ]:
q("""
SELECT
    COUNT(*) AS match_with_200plus,
    SUM(chase_won) AS successfully_chased 

    FROM v_match_totals
    WHERE first_innings_runs >=200;
    
""")

In [ ]:
q("""
SELECT
    MAX(CASE WHEN chase_won = 0 THEN first_innings_runs END) AS biggest_defend,
    MAX(CASE WHEN chase_won = 1 THEN first_innings_runs END) AS biggest_chased
FROM v_match_totals;
""")

In [ ]:
q("""

SELECT venue_clean, COUNT(*) AS matches_hosted
FROM matches_clean
GROUP BY venue_clean
ORDER BY matches_hosted DESC
LIMIT 10;

""")

In [ ]:
q("""
--which ground is the highest scoring?
SELECT m.venue_clean,
     count(DISTINCT m.match_id)as matches,\
     round(avg(i.runs),1) as avg_score
from matches_clean m
join v_innings i ON i.match_id=m.match_id

GROUP BY m.venue_clean
having matches>=25
ORDER BY avg_score DESC
LIMIT 10;


""")

In [ ]:
q("""

--which team has won the most marches?
select match_winner,count(*)as wins
from matches_clean
where result='win'
group by match_winner
order by wins desc;



""")